# Set Up Parser
The set up parser takes a JSON file that contains the number of qubits, their respective intial states statevectors represented as strings, an optional oracle, optional normalization option, and optional expected output.

The Parser will read the JSON file and return the number of qubits, intialized quantum circits for each qubit, and the oracle.

In [2]:
import json
import numpy as np
from QBitParser import Parser 
from qiskit import QuantumCircuit

def setup_parser(json_path):

    try:
        with open(json_path, "r") as file:
            setup = json.load(file)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Setup JSON file not found: {json_path}"
        )
    except json.JSONDecodeError as error:
        raise ValueError(
            f"Invalid JSON format in: {json_path}"
        ) from error

    #Check Structure
    if not isinstance(setup, dict):
        raise TypeError("Setup JSON must be a dictionary.")
    
    #Extract Values
    q_bit_count = setup.get("qbitcount", 1)
    initial_states = setup.get("initial states", "|0>")
    normalize = setup.get("normalize", False)
    output = setup.get("expected output", None)
    oracle = setup.get("oracle", None)

    q_bit_state_vectors = []

    if q_bit_count <= 0:
        raise TypeError("Q Bit Count Cannot be Negative or 0")

    if q_bit_count != 1:
        if not isinstance(initial_states, list):
            raise TypeError("Initial States must be a list")
        
        if any(not isinstance(initial, str) for initial in initial_states):
            raise TypeError("Items of Initial States must be string objects")
        
        if q_bit_count != len(initial_states):
            raise ValueError("Q-Bit Counts and Initial States Count Do Not Match")

        for initial in initial_states:
            #Parse Each Initial Value
            vector_value = Parser(initial).parse()
            #Append Actual QBit Circuit, with value, to list
            q_bit_state_vectors.append(QuantumCircuit(1).initialize(vector_value, 0, normalize = normalize))
    else:
        if not isinstance(initial_states, str):
            raise TypeError("Initial States must be a String Item")
        
        vector_value = Parser(initial_states).parse()
        #Append Actual QBit Circuit, with value, to list
        qc = QuantumCircuit(1)
        qc.initialize(vector_value, 0, normalize = normalize)
        q_bit_state_vectors.append(qc)

    return q_bit_count, q_bit_state_vectors, oracle, output

setup_parser("./test_files/setup_valid.json")

    

(1,
 None,
 None)